# AiStock V11.5 测试 Notebook — 7分量市场状态量化模型全面测试

## 版本升级表

| # | 分量 | 权重 | 引擎 | V11.5 变更 |
|---|------|------|------|------------|
| 1 | commodity | 0.20 | DerivativesSignalEngine | 无变更 |
| 2 | term_structure | 0.06 | DerivativesSignalEngine | 无变更 |
| 3 | index_basis | 0.16 | DerivativesSignalEngine | 无变更 |
| 4 | **fund_flow** | **0.10** | **FundFlowEngine** | **REACTIVATED** — 基于TDX成交量数据, 不使用akshare fund flow API |
| 5 | option_pcr | 0.10 | OptionPCREngine | 无变更 |
| 6 | macro_valuation | 0.10 | MacroValuationEngine | **PE-only, no PB**; online-first (AKAdapter → PostgreSQL fallback) |
| 7 | style_rotation | 0.28 | StyleRotationEngine | **momentum+valuation dual-factor**; 行业轮动含PE估值 |

### V11.5 关键变更摘要
- **FundFlowEngine REACTIVATED**: 从 DEPRECATED 升级为正式分量, 数据源仅使用TDX成交量
- **MacroValuationEngine**: PE-only (移除PB), online-first 架构
- **StyleRotationEngine**: 行业轮动升级为动量+估值双因子驱动
- **PE百分位**: 从历史PE序列计算 (window=2500)
- **数据库列名**: 中文 (日期, 滚动市盈率, etc.)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1: 环境设置 — 添加项目根目录到 sys.path, 导入核心模块
# ═══════════════════════════════════════════════════════════════════════
import sys
import os
import logging

# 添加项目根目录到 sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"📂 PROJECT_ROOT: {PROJECT_ROOT}")
print(f"📂 sys.path[0]: {sys.path[0]}")

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(name)s] %(levelname)s: %(message)s',
    datefmt='%H:%M:%S'
)

# 导入核心模块并验证
try:
    from base_service.config_service import ConfigService
    print("✅ ConfigService 导入成功")
except ImportError as e:
    print(f"❌ ConfigService 导入失败: {e}")

try:
    from base_service.cache_service import CacheService
    print("✅ CacheService 导入成功")
except ImportError as e:
    print(f"❌ CacheService 导入失败: {e}")

try:
    from base_service.event_bus import EventBus, Topics
    print("✅ EventBus 导入成功")
except ImportError as e:
    print(f"❌ EventBus 导入失败: {e}")

try:
    from base_service.service_container import ServiceContainer, SubsystemBase
    print("✅ ServiceContainer + SubsystemBase 导入成功")
except ImportError as e:
    print(f"❌ ServiceContainer 导入失败: {e}")

try:
    from data_service.tdx_adapter import TDXAdapter, MarketType, BarCategory
    print("✅ TDXAdapter 导入成功")
except ImportError as e:
    print(f"❌ TDXAdapter 导入失败: {e}")

try:
    from data_service.ak_adapter import AKAdapter
    print("✅ AKAdapter 导入成功")
except ImportError as e:
    print(f"❌ AKAdapter 导入失败: {e}")

try:
    from data_service.database_reader import DatabaseReader
    print("✅ DatabaseReader 导入成功")
except ImportError as e:
    print(f"❌ DatabaseReader 导入失败: {e}")

try:
    from data_service.data_loader_service import DataLoaderService
    print("✅ DataLoaderService 导入成功")
except ImportError as e:
    print(f"❌ DataLoaderService 导入失败: {e}")

try:
    from subsystems.market_state.core.fund_flow_engine import FundFlowEngine, FundFlowSignal
    print("✅ FundFlowEngine (V11.5 REACTIVATED) 导入成功")
except ImportError as e:
    print(f"❌ FundFlowEngine 导入失败: {e}")

try:
    from subsystems.market_state.core.macro_valuation_engine import MacroValuationEngine, MacroValuationSignal
    print("✅ MacroValuationEngine (V11.5 PE-only) 导入成功")
except ImportError as e:
    print(f"❌ MacroValuationEngine 导入失败: {e}")

try:
    from subsystems.market_state.core.style_rotation_engine import StyleRotationEngine, StyleRotationSignal, IndustryRotationSignal
    print("✅ StyleRotationEngine (V11.5 dual-factor) 导入成功")
except ImportError as e:
    print(f"❌ StyleRotationEngine 导入失败: {e}")

try:
    from subsystems.market_state.core.option_pcr_engine import OptionPCREngine, OptionPCRSignal
    print("✅ OptionPCREngine 导入成功")
except ImportError as e:
    print(f"❌ OptionPCREngine 导入失败: {e}")

try:
    from subsystems.market_state.core.derivatives_signal_engine import DerivativesSignalEngine, DerivativesResult
    print("✅ DerivativesSignalEngine 导入成功")
except ImportError as e:
    print(f"❌ DerivativesSignalEngine 导入失败: {e}")

try:
    from subsystems.market_state.core.market_state_engine import MarketStateEngine
    print("✅ MarketStateEngine 导入成功")
except ImportError as e:
    print(f"❌ MarketStateEngine 导入失败: {e}")

# 验证配置文件路径
config_dir = os.path.join(PROJECT_ROOT, 'config', 'yaml')
yaml_files = sorted([f for f in os.listdir(config_dir) if f.endswith('.yaml')])
print(f"\n📂 配置目录: {config_dir}")
print(f"📄 YAML 文件 ({len(yaml_files)}): {yaml_files}")

# 验证数据目录
data_dir = os.path.join(PROJECT_ROOT, 'data')
if os.path.exists(data_dir):
    data_files = os.listdir(data_dir)
    print(f"📊 数据文件: {data_files[:5]}...")
else:
    print(f"⚠️ 数据目录不存在: {data_dir}")

---
## Module 1: Infrastructure Layer Tests

测试基础设施层: ConfigService / CacheService / EventBus / ServiceContainer + SubsystemBase

这些是V10架构的核心基础设施, 为V11.5 7分量系统提供支撑。

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1.1 ConfigService — 加载8个YAML文件, 验证V11.5 7分量权重
# ═══════════════════════════════════════════════════════════════════════
from base_service.config_service import ConfigService

config_dir_path = os.path.join(PROJECT_ROOT, 'config', 'yaml')
config = ConfigService(config_dir=config_dir_path, enable_hot_reload=False)
config.load_all()

# 验证加载的配置文件数量
loaded_files = list(config._configs.keys())
print(f"✅ 已加载 {len(loaded_files)} 个配置文件: {loaded_files}")
assert len(loaded_files) >= 7, f"期望至少7个配置文件, 实际 {len(loaded_files)}"

# ── 验证 V11.5 7分量 composite_weights ──────────────────────────
composite_weights = config.get("market_state.composite_weights", {})
print(f"\n📊 V11.5 composite_weights: {composite_weights}")

expected_keys = {"commodity", "term_structure", "index_basis", "fund_flow", "option_pcr", "macro_valuation", "style_rotation"}
actual_keys = set(composite_weights.keys())
assert actual_keys == expected_keys, f"composite_weights 键不匹配: 期望 {expected_keys}, 实际 {actual_keys}"
print(f"✅ composite_weights 包含 7 个分量: {actual_keys}")

# 验证 fund_flow 权重 = 0.10 (V11.5 REACTIVATED)
fund_flow_weight = composite_weights.get("fund_flow", 0)
assert fund_flow_weight == 0.10, f"fund_flow 权重应为 0.10, 实际 {fund_flow_weight}"
print(f"✅ fund_flow 权重 = {fund_flow_weight} (V11.5 REACTIVATED)")

# 验证权重之和 ≈ 1.0
total_weight = sum(composite_weights.values())
assert abs(total_weight - 1.0) < 0.001, f"权重之和应为 1.0, 实际 {total_weight}"
print(f"✅ 权重之和 = {total_weight:.2f}")

# ── 验证 PE-only 配置 (V11.5) ─────────────────────────────────
derived_config = config.get("database.csi_index_pe.derived_columns", {})
pe_config = derived_config.get("pe_percentile", {})
print(f"\n📊 PE percentile 配置: {pe_config}")
assert pe_config.get("source") == "pe_ttm", f"PE percentile source 应为 pe_ttm"
assert pe_config.get("window") == 2500, f"PE percentile window 应为 2500"
print(f"✅ PE-only 配置: source=pe_ttm, window=2500, method={pe_config.get('method', 'rank')}")

# 验证无 PB 相关配置
pb_in_derived = "pb_percentile" in derived_config
print(f"{'❌' if pb_in_derived else '✅'} derived_columns 中 pb_percentile {'存在(不应存在!)' if pb_in_derived else '不存在(正确)'}")

# ── 验证 fund_flow_weights (V11.5 REACTIVATED) ────────────────
fund_flow_weights = config.get("market_state.fund_flow_weights", {})
print(f"\n📊 fund_flow_weights: {fund_flow_weights}")
expected_ff_keys = {"volume_trend", "size_rotation", "leverage_ratio", "momentum"}
assert set(fund_flow_weights.keys()) == expected_ff_keys, f"fund_flow_weights 键不匹配"
ff_total = sum(fund_flow_weights.values())
assert abs(ff_total - 1.0) < 0.001, f"fund_flow 权重之和应为 1.0, 实际 {ff_total}"
print(f"✅ fund_flow_weights: 4个子信号, 权重之和 = {ff_total:.2f}")

# ── 验证 industry_valuation_weights (V11.5 NEW) ──────────────
industry_val_weights = config.get("market_state.industry_valuation_weights", {})
print(f"\n📊 industry_valuation_weights: {industry_val_weights}")
assert "momentum" in industry_val_weights, "industry_valuation_weights 缺少 momentum"
assert "valuation" in industry_val_weights, "industry_valuation_weights 缺少 valuation"
iv_total = sum(industry_val_weights.values())
assert abs(iv_total - 1.0) < 0.001, f"industry_valuation 权重之和应为 1.0, 实际 {iv_total}"
print(f"✅ industry_valuation_weights: momentum={industry_val_weights.get('momentum')}, valuation={industry_val_weights.get('valuation')}")

print("\n🎉 1.1 ConfigService V11.5 配置验证全部通过!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1.2 CacheService — 命名空间隔离, TTL, 统计信息
# ═══════════════════════════════════════════════════════════════════════
from base_service.cache_service import CacheService

cache = CacheService(default_ttl=60, default_max_size=100)

# 1.2.1 命名空间隔离测试
cache.set("key1", "value_data", namespace="data")
cache.set("key1", "value_config", namespace="config")
cache.set("key1", "value_computation", namespace="computation")

v_data = cache.get("key1", namespace="data")
v_config = cache.get("key1", namespace="config")
v_comp = cache.get("key1", namespace="computation")

assert v_data == "value_data", f"data 命名空间隔离失败"
assert v_config == "value_config", f"config 命名空间隔离失败"
assert v_comp == "value_computation", f"computation 命名空间隔离失败"
print(f"✅ 命名空间隔离: data={v_data}, config={v_config}, computation={v_comp}")

# 1.2.2 代理对象测试
ns_data = cache.namespace("data")
ns_data.set("k2", "v2")
ns_data.set_batch({"k3": "v3", "k4": "v4"})
batch_result = ns_data.get_batch(["k2", "k3", "k4"])
assert len(batch_result) == 3, f"batch get 应返回3个结果, 实际 {len(batch_result)}"
print(f"✅ 命名空间代理: set_batch/get_batch 成功, 获取 {len(batch_result)} 个值")

# 1.2.3 TTL 过期测试
import time
cache.set("short_lived", "expires_soon", ttl=1, namespace="data")
v_before = cache.get("short_lived", namespace="data")
assert v_before == "expires_soon", "TTL 未过期前应能获取"
time.sleep(1.5)
v_after = cache.get("short_lived", namespace="data")
assert v_after is None, "TTL 过期后应返回 None"
print(f"✅ TTL 过期: 过期前={v_before}, 过期后={v_after}")

# 1.2.4 统计信息测试
stats = cache.get_stats()
print(f"✅ 缓存统计: 命名空间数={len(stats)}")
for ns_name, ns_stats in stats.items():
    print(f"   - {ns_name}: size={ns_stats.get('size', 0)}, hits={ns_stats.get('hits', 0)}, misses={ns_stats.get('misses', 0)}")

# 1.2.5 预创建命名空间验证
precreated = {"config", "data", "computation", "contract"}
actual_ns = set(stats.keys())
assert precreated.issubset(actual_ns), f"预创建命名空间缺失: {precreated - actual_ns}"
print(f"✅ 预创建命名空间: {precreated} 均存在")

print("\n🎉 1.2 CacheService 测试全部通过!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1.3 EventBus — 发布/订阅, 通配符匹配, 历史回放
# ═══════════════════════════════════════════════════════════════════════
from base_service.event_bus import EventBus, Topics, Event

bus = EventBus(history_size=50)

# 1.3.1 精确匹配发布/订阅
received_events = []

def on_market_state_updated(event: Event):
    received_events.append(event)
    print(f"   📩 收到事件: {event.topic} | data={event.data}")

bus.subscribe(Topics.MARKET_STATE_UPDATED, on_market_state_updated)

# 发布事件
bus.publish(Topics.MARKET_STATE_UPDATED, {
    "composite_signal": 35.2,
    "composite_direction": "bullish",
    "model": "7-component",
    "version": "11.5",
})

assert len(received_events) == 1, f"应收到1个事件, 实际 {len(received_events)}"
assert received_events[0].data["model"] == "7-component"
print(f"✅ 精确匹配: 收到事件, model={received_events[0].data['model']}, version={received_events[0].data['version']}")

# 1.3.2 通配符匹配
wildcard_events = []

def on_market_state_wildcard(event: Event):
    wildcard_events.append(event)

bus.subscribe("market_state.*", on_market_state_wildcard)

bus.publish(Topics.MARKET_STATE_UPDATED, {"test": "wildcard_1"})
bus.publish("market_state.warning", {"test": "wildcard_2"})

assert len(wildcard_events) >= 2, f"通配符应匹配至少2个事件, 实际 {len(wildcard_events)}"
print(f"✅ 通配符匹配: 收到 {len(wildcard_events)} 个 market_state.* 事件")

# 1.3.3 多层通配符
multi_wildcard_events = []

def on_market_state_multi(event: Event):
    multi_wildcard_events.append(event)

bus.subscribe("market_state.**", on_market_state_multi)

bus.publish("market_state.updated.nested", {"test": "multi_wildcard"})
assert len(multi_wildcard_events) >= 1, f"多层通配符应匹配"
print(f"✅ 多层通配符: 收到 {len(multi_wildcard_events)} 个 market_state.** 事件")

# 1.3.4 历史回放
replay_count = [0]

def on_replay(event: Event):
    replay_count[0] += 1

bus.replay(Topics.MARKET_STATE_UPDATED, on_replay, limit=5)
print(f"✅ 历史回放: 回放了 {replay_count[0]} 个 {Topics.MARKET_STATE_UPDATED} 事件")

# 1.3.5 预定义主题常量验证
print(f"✅ 预定义主题: CONFIG_CHANGED={Topics.CONFIG_CHANGED}, DATA_LOADED={Topics.DATA_LOADED}")
print(f"   MARKET_STATE_UPDATED={Topics.MARKET_STATE_UPDATED}, SUBSYSTEM_STARTED={Topics.SUBSYSTEM_STARTED}")

print("\n🎉 1.3 EventBus 测试全部通过!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1.4 ServiceContainer + SubsystemBase — 依赖注入
# ═══════════════════════════════════════════════════════════════════════
from base_service.service_container import ServiceContainer, SubsystemBase

container = ServiceContainer()

# 1.4.1 单例注册
container.register_singleton("config", config)
container.register_singleton("cache", cache)
container.register_singleton("event_bus", bus)

assert container.has("config"), "config 服务应已注册"
assert container.has("cache"), "cache 服务应已注册"
assert container.has("event_bus"), "event_bus 服务应已注册"
print(f"✅ 单例注册: config={container.has('config')}, cache={container.has('cache')}, event_bus={container.has('event_bus')}")

# 1.4.2 工厂注册 (懒加载)
factory_called = [False]

def create_expensive_service():
    factory_called[0] = True
    return {"name": "expensive_service", "created_at": time.time()}

container.register_factory("expensive", create_expensive_service)
assert container.has("expensive"), "expensive 工厂应已注册"
assert not factory_called[0], "工厂不应在注册时调用"
print(f"✅ 工厂注册: 已注册但未实例化 (lazy)")

# 触发懒加载
expensive = container.get("expensive")
assert factory_called[0], "工厂应在获取时调用"
assert expensive["name"] == "expensive_service"
print(f"✅ 懒加载: 工厂在 get() 时调用, 服务={expensive['name']}")

# 1.4.3 服务列表
services = container.list_services()
print(f"✅ 已注册服务: {services}")

# 1.4.4 SubsystemBase 自动注入测试
class TestSubsystem(SubsystemBase):
    def __init__(self, container: ServiceContainer):
        super().__init__("test_subsystem", container)
        
test_sub = TestSubsystem(container)
assert test_sub.config is config, "SubsystemBase 应自动注入 config"
assert test_sub.cache is cache, "SubsystemBase 应自动注入 cache"
assert test_sub.event_bus is bus, "SubsystemBase 应自动注入 event_bus"
print(f"✅ SubsystemBase 自动注入: config/cache/event_bus 均已注入")

# 1.4.5 子系统配置隔离
sub_config = test_sub.subsystem_config
assert isinstance(sub_config, dict), "子系统配置应为 dict"
composite_in_sub = sub_config.get("composite_weights", {})
assert "fund_flow" in composite_in_sub, "子系统配置应包含 fund_flow (V11.5)"
print(f"✅ 子系统配置隔离: composite_weights 包含 fund_flow={composite_in_sub.get('fund_flow')}")

# 1.4.6 异常测试 — 未注册服务
try:
    container.get("nonexistent_service")
    print("❌ 未注册服务应抛出 KeyError")
except KeyError as e:
    print(f"✅ 未注册服务正确抛出 KeyError: {e}")

print("\n🎉 1.4 ServiceContainer + SubsystemBase 测试全部通过!")

---
## Module 2: Data Service Layer Tests

测试数据服务层: TDXAdapter / AKAdapter / DatabaseReader / DataLoaderService

### V11.5 关键变更
- **DatabaseReader**: PE-only (移除PB), online-first 架构
- **AKAdapter**: 新增 `get_index_pe_csindex()` 方法获取中证指数PE
- **中文列名**: 数据库列名为中文 (日期, 滚动市盈率, etc.)
- **FundFlowEngine**: 直接使用TDX成交量数据, 不通过DataLoaderService

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2.1 TDXAdapter — 双端口健康检查, 指数/期货K线
# ═══════════════════════════════════════════════════════════════════════
from data_service.tdx_adapter import TDXAdapter, MarketType, BarCategory

# 使用 ConfigService 初始化 TDXAdapter
try:
    tdx = TDXAdapter(config_service=config)
    print("✅ TDXAdapter 初始化成功 (ConfigService)")
except Exception as e:
    tdx = TDXAdapter()  # 回退到默认配置
    print(f"⚠️ TDXAdapter 初始化使用默认配置: {e}")

# 双端口健康检查
try:
    health = tdx.health_check()
    std_ok = health.get("standard_port", False)
    ext_ok = health.get("extension_port", False)
    print(f"{'✅' if std_ok else '❌'} 标准端口 (7709): {std_ok}")
    print(f"{'✅' if ext_ok else '❌'} 扩展端口 (7721): {ext_ok}")
except Exception as e:
    print(f"⚠️ TDXAdapter 健康检查失败 (TDX可能离线): {e}")
    std_ok = False
    ext_ok = False

# 测试获取指数K线 (标准端口)
if std_ok:
    try:
        df_hs300 = tdx.get_index_daily(code="000300", market_type=MarketType.INDEX_SH, count=10)
        if not df_hs300.empty:
            print(f"✅ 沪深300 K线: {len(df_hs300)} 条, 列={list(df_hs300.columns)[:6]}")
        else:
            print(f"⚠️ 沪深300 K线数据为空")
    except Exception as e:
        print(f"⚠️ 沪深300 K线获取失败: {e}")
else:
    print("⚠️ TDX标准端口不可用, 跳过指数K线测试")

# 测试获取期货K线 (扩展端口)
if ext_ok:
    try:
        df_if = tdx.get_future_daily(code="IFL8", market_type=MarketType.FUTURE_ZJ, count=10)
        if not df_if.empty:
            print(f"✅ IF主连 K线: {len(df_if)} 条, 列={list(df_if.columns)[:6]}")
        else:
            print(f"⚠️ IF主连 K线数据为空")
    except Exception as e:
        print(f"⚠️ IF主连 K线获取失败: {e}")
else:
    print("⚠️ TDX扩展端口不可用, 跳过期货K线测试")

# 测试宏观指标 (扩展端口 Market 38)
if ext_ok:
    try:
        df_pmi = tdx.get_macro_data(code="3_PMI", market=38, count=10)
        if not df_pmi.empty:
            print(f"✅ 制造业PMI: {len(df_pmi)} 条")
        else:
            print(f"⚠️ PMI 数据为空")
    except Exception as e:
        print(f"⚠️ PMI 获取失败: {e}")
else:
    print("⚠️ TDX扩展端口不可用, 跳过宏观数据测试")

print("\n🎉 2.1 TDXAdapter 测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2.2 AKAdapter — 健康检查, 海外期货, V11.5 get_index_pe_csindex()
# ═══════════════════════════════════════════════════════════════════════
from data_service.ak_adapter import AKAdapter

try:
    ak_adapter = AKAdapter(config_service=config)
    print("✅ AKAdapter 初始化成功 (ConfigService)")
except Exception as e:
    ak_adapter = AKAdapter()
    print(f"⚠️ AKAdapter 初始化使用默认配置: {e}")

# 测试海外期货 (CORE层)
try:
    core_data = ak_adapter.get_futures_by_tier("core")
    core_count = len(core_data)
    print(f"{'✅' if core_count > 0 else '⚠️'} CORE 海外期货: {core_count} 品种")
    for sym, data in list(core_data.items())[:3]:
        print(f"   - {sym}: {data.get('name', '?')} price={data.get('price', 0):.2f}")
except Exception as e:
    print(f"⚠️ 海外期货获取失败 (akshare可能不可用): {e}")

# ── V11.5 新增: get_index_pe_csindex() 测试 ─────────────────────
print("\n--- V11.5 get_index_pe_csindex() 测试 ---")
try:
    # 测试沪深300 PE
    df_pe_300 = ak_adapter.get_index_pe_csindex("000300", start_date="20240101")
    if df_pe_300 is not None and not df_pe_300.empty:
        print(f"✅ 沪深300 PE数据: {len(df_pe_300)} 条")
        print(f"   列名: {list(df_pe_300.columns)[:8]}")
        if "pe_ttm" in df_pe_300.columns:
            latest_pe = df_pe_300.iloc[0].get("pe_ttm", 0)
            print(f"   最新PE_TTM: {latest_pe}")
    else:
        print(f"⚠️ 沪深300 PE数据为空 (akshare可能不可用)")
except Exception as e:
    print(f"⚠️ get_index_pe_csindex() 测试失败: {e}")

print("\n🎉 2.2 AKAdapter 测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2.3 DatabaseReader — V11.5 PE-only, online-first
# ═══════════════════════════════════════════════════════════════════════
from data_service.database_reader import DatabaseReader

# 使用 csi_index_pe 引擎 (中文列名)
try:
    db_reader = DatabaseReader(
        config_service=config,
        engine_name="csi_index_pe",
    )
    is_connected = getattr(db_reader, 'is_connected', False) or db_reader._connected
    print(f"{'✅' if is_connected else '❌'} DatabaseReader 连接状态: {is_connected}")
    print(f"   table_mode={db_reader._table_mode}, column_lang={db_reader._column_lang}")
    print(f"   pe_columns keys: {list(db_reader._pe_columns.keys())[:5]}")
except Exception as e:
    db_reader = None
    is_connected = False
    print(f"⚠️ DatabaseReader 初始化失败: {e}")

if is_connected and db_reader is not None:
    # 2.3.1 测试 get_index_pe() — PE-only
    try:
        df_pe = db_reader.get_index_pe("000300", days=30)
        if not df_pe.empty:
            cols = list(df_pe.columns)
            has_pb = "pb" in cols or "pb_percentile" in cols
            has_pe = "pe_ttm" in cols
            has_pe_pct = "pe_percentile" in cols
            print(f"{'✅' if has_pe else '❌'} get_index_pe() 包含 pe_ttm: {has_pe}")
            print(f"{'✅' if has_pe_pct else '❌'} get_index_pe() 包含 pe_percentile: {has_pe_pct}")
            print(f"{'✅' if not has_pb else '❌'} get_index_pe() 不含 PB 字段: {not has_pb} (V11.5 PE-only)")
            print(f"   数据: {len(df_pe)} 行, 列={cols}")
        else:
            print(f"⚠️ 000300 PE数据为空")
    except Exception as e:
        print(f"⚠️ get_index_pe() 失败: {e}")

    # 2.3.2 测试 get_index_pe_online_first() — V11.5 online-first
    try:
        df_pe_online = db_reader.get_index_pe_online_first(
            "000300", days=30, ak_adapter=ak_adapter
        )
        if not df_pe_online.empty:
            print(f"✅ get_index_pe_online_first(): {len(df_pe_online)} 行")
            print(f"   列={list(df_pe_online.columns)}")
        else:
            print(f"⚠️ online_first 数据为空")
    except Exception as e:
        print(f"⚠️ get_index_pe_online_first() 失败: {e}")

    # 2.3.3 测试 get_index_kline()
    try:
        df_kline = db_reader.get_index_kline("000300", days=10)
        if not df_kline.empty:
            print(f"✅ get_index_kline(): {len(df_kline)} 行")
        else:
            print(f"⚠️ kline 数据为空")
    except Exception as e:
        print(f"⚠️ get_index_kline() 失败: {e}")

    # 2.3.4 测试 get_valuation_percentiles() — V11.5 PE-only
    try:
        val_pct = db_reader.get_valuation_percentiles("000300", days=500)
        has_pb_pct = "pb_percentile" in val_pct
        has_pb_field = "has_pb" in val_pct
        print(f"{'✅' if not has_pb_pct else '❌'} get_valuation_percentiles() 不含 pb_percentile: {not has_pb_pct}")
        print(f"{'✅' if not has_pb_field else '❌'} get_valuation_percentiles() 不含 has_pb: {not has_pb_field}")
        print(f"   pe_percentile={val_pct.get('pe_percentile', 'N/A')}")
        print(f"   data_available={val_pct.get('data_available', False)}")
    except Exception as e:
        print(f"⚠️ get_valuation_percentiles() 失败: {e}")
else:
    print("⚠️ PostgreSQL 未连接, 跳过 DatabaseReader 测试")
    print("   如需测试, 请确保 PostgreSQL csiIndexPE 数据库可访问")

print("\n🎉 2.3 DatabaseReader 测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2.4 AKAdapter PE Test — V11.5 get_index_pe_csindex() 深度测试
# ═══════════════════════════════════════════════════════════════════════

pe_test_codes = ["000300", "000905", "932083"]
pe_results = {}

for code in pe_test_codes:
    try:
        df = ak_adapter.get_index_pe_csindex(code, start_date="20200101")
        if df is not None and not df.empty:
            # 检查关键列
            has_trade_date = "trade_date" in df.columns
            has_pe_ttm = "pe_ttm" in df.columns
            has_close = "close" in df.columns
            
            # 获取最新数据
            latest = df.iloc[0]
            pe_ttm_val = latest.get("pe_ttm", None)
            close_val = latest.get("close", None)
            trade_date_val = latest.get("trade_date", None)
            
            pe_results[code] = {
                "rows": len(df),
                "has_trade_date": has_trade_date,
                "has_pe_ttm": has_pe_ttm,
                "has_close": has_close,
                "latest_pe_ttm": pe_ttm_val,
                "latest_close": close_val,
                "latest_date": str(trade_date_val),
            }
            
            print(f"✅ {code}: {len(df)} 行 | trade_date={has_trade_date} pe_ttm={has_pe_ttm} close={has_close}")
            print(f"   最新: date={trade_date_val}, PE_TTM={pe_ttm_val}, close={close_val}")
            
            # 计算PE百分位 (V11.5)
            if has_pe_ttm:
                try:
                    from data_service.database_reader import _calc_percentile_rank
                    import pandas as pd
                    pe_series = pd.to_numeric(df["pe_ttm"], errors="coerce").dropna()
                    if len(pe_series) >= 10:
                        pct_series = _calc_percentile_rank(pe_series, 2500)
                        latest_pct = float(pct_series.iloc[0])
                        print(f"   V11.5 PE百分位 (window=2500): {latest_pct:.2f}%")
                        pe_results[code]["pe_percentile"] = latest_pct
                except Exception as e:
                    print(f"   ⚠️ PE百分位计算失败: {e}")
        else:
            print(f"⚠️ {code}: 数据为空 (akshare可能不支持此指数)")
            pe_results[code] = {"rows": 0}
    except Exception as e:
        print(f"⚠️ {code}: get_index_pe_csindex() 失败: {e}")
        pe_results[code] = {"error": str(e)}

# 汇总
success_codes = [c for c, r in pe_results.items() if r.get("rows", 0) > 0]
print(f"\n📊 PE测试汇总: {len(success_codes)}/{len(pe_test_codes)} 指数数据可用")
print("\n🎉 2.4 AKAdapter PE 深度测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2.5 DataLoaderService — 数据加载编排
# ═══════════════════════════════════════════════════════════════════════
from data_service.data_loader_service import DataLoaderService

try:
    data_loader = DataLoaderService(
        tdx_adapter=tdx,
        ak_adapter=ak_adapter,
        db_reader=db_reader if db_reader else DatabaseReader(config_service=config),
        config_service=config,
    )
    print("✅ DataLoaderService 初始化成功")
    
    # 查看配置加载结果
    print(f"   指数代码: {len(data_loader._index_codes)}")
    print(f"   期货代码: {len(data_loader._future_codes)}")
    print(f"   期权标的: {len(data_loader._option_underlyings_list)}")
except Exception as e:
    data_loader = None
    print(f"⚠️ DataLoaderService 初始化失败: {e}")

# 测试按段加载 (仅加载指数段, 避免全量加载耗时)
if data_loader is not None:
    try:
        from data_service.data_loader_service import DataSection
        index_data = data_loader.load_section(DataSection.INDEX_DATA)
        loaded_count = sum(1 for v in index_data.values() if v is not None and not v.empty)
        total_count = len(index_data)
        print(f"✅ 指数数据加载: {loaded_count}/{total_count} 成功")
    except Exception as e:
        print(f"⚠️ 指数数据加载失败: {e}")

    # 测试海外期货加载 (CORE层)
    try:
        overseas = data_loader.load_overseas_futures(tier="core")
        overseas_count = len(overseas) if overseas else 0
        print(f"{'✅' if overseas_count > 0 else '⚠️'} 海外期货 (CORE): {overseas_count} 品种")
    except Exception as e:
        print(f"⚠️ 海外期货加载失败: {e}")

print("\n🎉 2.5 DataLoaderService 测试完成!")

---
## Module 3: V11.5 Engine Tests

### V11.5 关键变更
- **FundFlowEngine**: REACTIVATED — 基于TDX成交量数据, 不使用akshare fund flow API
- **MacroValuationEngine**: PE-only (移除PB), online-first (AKAdapter → PostgreSQL fallback)
- **StyleRotationEngine**: 行业轮动升级为 momentum+valuation dual-factor

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3.1 FundFlowEngine — V11.5 REACTIVATED (基于TDX成交量数据)
# ═══════════════════════════════════════════════════════════════════════
from subsystems.market_state.core.fund_flow_engine import FundFlowEngine, FundFlowSignal

print("--- FundFlowEngine V11.5 REACTIVATED 测试 ---")

# 初始化 FundFlowEngine
ff_engine = FundFlowEngine(
    tdx_adapter=tdx,
    config=config,
    cache=cache,
)
print(f"✅ FundFlowEngine 初始化成功")
print(f"   大盘指数: {ff_engine._large_cap_codes}")
print(f"   小盘指数: {ff_engine._small_cap_codes}")
print(f"   股指期货: {ff_engine._futures_codes}")
print(f"   权重: {ff_engine._weights}")

# 验证4个子信号权重
expected_sub_signals = {"volume_trend", "size_rotation", "leverage_ratio", "momentum"}
actual_sub_signals = set(ff_engine._weights.keys())
assert actual_sub_signals == expected_sub_signals, f"子信号不匹配: {actual_sub_signals}"
print(f"✅ 4个子信号: {actual_sub_signals}")

# 验证不使用 akshare fund flow API
ff_source_code = open(os.path.join(PROJECT_ROOT, 'subsystems/market_state/core/fund_flow_engine.py'), 'r').read()
uses_akshare_ff = 'ak.stock_individual_fund_flow' in ff_source_code or 'ak.stock_market_fund_flow' in ff_source_code
print(f"{'✅' if not uses_akshare_ff else '❌'} 不使用 akshare fund flow API: {not uses_akshare_ff} (V11.5)")

# 执行计算
try:
    ff_signal = ff_engine.calculate()
    print(f"\n📊 FundFlowSignal 计算:")
    print(f"   composite_signal: {ff_signal.composite_signal:.2f}")
    print(f"   composite_direction: {ff_signal.composite_direction}")
    print(f"   data_available: {ff_signal.data_available}")
    print(f"   volume_trend_signal: {ff_signal.volume_trend_signal:.2f}")
    print(f"   size_rotation_signal: {ff_signal.size_rotation_signal:.2f}")
    print(f"   leverage_signal: {ff_signal.leverage_signal:.2f}")
    print(f"   momentum_signal: {ff_signal.momentum_signal:.2f}")
    
    # 验证信号范围
    assert -100 <= ff_signal.composite_signal <= 100, f"综合信号超出范围: {ff_signal.composite_signal}"
    assert ff_signal.composite_direction in ("bullish", "bearish", "neutral")
    print(f"✅ 信号范围验证通过")
    
    # 验证 FundFlowSignal 数据类字段
    ff_dict = ff_signal.to_dict()
    print(f"\n📋 FundFlowSignal.to_dict():")
    for k, v in ff_dict.items():
        if k != "snapshot":
            print(f"   {k}: {v}")
    print(f"   snapshot: {ff_dict.get('snapshot', {})}")
    
except Exception as e:
    print(f"⚠️ FundFlowEngine 计算失败 (TDX可能离线): {e}")

print("\n🎉 3.1 FundFlowEngine V11.5 测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3.2 MacroValuationEngine — V11.5 PE-only, online-first
# ═══════════════════════════════════════════════════════════════════════
from subsystems.market_state.core.macro_valuation_engine import MacroValuationEngine, MacroValuationSignal

print("--- MacroValuationEngine V11.5 PE-only 测试 ---")

# 初始化 MacroValuationEngine
mv_engine = MacroValuationEngine(
    tdx_adapter=tdx,
    db_reader=db_reader,
    ak_adapter=ak_adapter,
    config=config,
    cache=cache,
)
print(f"✅ MacroValuationEngine 初始化成功")
print(f"   权重: {mv_engine._weights}")
print(f"   估值代码: {mv_engine._valuation_codes}")

# 验证 PE-only: 检查 MacroValuationSignal 无 PB 字段
mv_signal_test = MacroValuationSignal()
mv_fields = [f for f in dir(mv_signal_test) if not f.startswith('_')]
has_pb = 'pb_percentile' in mv_fields or 'pb_signal' in mv_fields
has_pe_pct = 'pe_percentile_avg' in mv_fields
print(f"{'✅' if not has_pb else '❌'} MacroValuationSignal 无 PB 字段: {not has_pb}")
print(f"{'✅' if has_pe_pct else '❌'} MacroValuationSignal 包含 pe_percentile_avg: {has_pe_pct}")

# 验证 V11.5 PE-only 源码
mv_source = open(os.path.join(PROJECT_ROOT, 'subsystems/market_state/core/macro_valuation_engine.py'), 'r').read()
has_pb_in_source = 'pb_percentile' in mv_source and 'pe_percentile' not in mv_source
print(f"{'✅' if not has_pb_in_source else '⚠️'} 源码中 PE 逻辑为主 (V11.5 PE-only)")

# 验证 online-first 逻辑
has_online_first = 'get_index_pe_online_first' in mv_source or '_get_pe_percentile_for_code' in mv_source
print(f"{'✅' if has_online_first else '❌'} 包含 online-first PE 获取逻辑: {has_online_first}")

# 执行计算
try:
    mv_signal = mv_engine.calculate()
    print(f"\n📊 MacroValuationSignal 计算:")
    print(f"   composite_signal: {mv_signal.composite_signal:.2f}")
    print(f"   composite_direction: {mv_signal.composite_direction}")
    print(f"   data_available: {mv_signal.data_available}")
    print(f"   pmi_signal: {mv_signal.pmi_signal:.2f}")
    print(f"   credit_signal: {mv_signal.credit_signal:.2f}")
    print(f"   liquidity_signal: {mv_signal.liquidity_signal:.2f}")
    print(f"   inflation_signal: {mv_signal.inflation_signal:.2f}")
    print(f"   valuation_signal: {mv_signal.valuation_signal:.2f} (PE only)")
    print(f"   pe_percentile_avg: {mv_signal.pe_percentile_avg:.2f}")
    
    # 验证信号范围
    assert -100 <= mv_signal.composite_signal <= 100
    print(f"✅ 信号范围验证通过")
    
    # to_dict 测试
    mv_dict = mv_signal.to_dict()
    has_pb_in_output = 'pb_percentile' in str(mv_dict)
    print(f"{'✅' if not has_pb_in_output else '❌'} to_dict() 无 PB 字段: {not has_pb_in_output}")
    
except Exception as e:
    print(f"⚠️ MacroValuationEngine 计算失败: {e}")

print("\n🎉 3.2 MacroValuationEngine V11.5 测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3.3 StyleRotationEngine — V11.5 momentum+valuation dual-factor
# ═══════════════════════════════════════════════════════════════════════
from subsystems.market_state.core.style_rotation_engine import StyleRotationEngine, StyleRotationSignal, IndustryRotationSignal

print("--- StyleRotationEngine V11.5 momentum+valuation dual-factor 测试 ---")

# 初始化 StyleRotationEngine
sr_engine = StyleRotationEngine(
    tdx_adapter=tdx,
    ak_adapter=ak_adapter,
    db_reader=db_reader,
    config=config,
    cache=cache,
)
print(f"✅ StyleRotationEngine 初始化成功")
print(f"   行业指数: {len(sr_engine._industry_indices)}")
print(f"   风格指数: {len(sr_engine._style_indices)}")
print(f"   规模指数: {len(sr_engine._size_indices)}")
print(f"   行业估值权重: {sr_engine._industry_val_weights}")

# 验证 V11.5: IndustryRotationSignal 包含 pe_ttm, pe_percentile, valuation_signal
irs_test = IndustryRotationSignal()
irs_fields = [f for f in dir(irs_test) if not f.startswith('_')]
has_pe_ttm = 'pe_ttm' in irs_fields
has_pe_pct = 'pe_percentile' in irs_fields
has_val_sig = 'valuation_signal' in irs_fields
print(f"{'✅' if has_pe_ttm else '❌'} IndustryRotationSignal.pe_ttm: {has_pe_ttm} (V11.5)")
print(f"{'✅' if has_pe_pct else '❌'} IndustryRotationSignal.pe_percentile: {has_pe_pct} (V11.5)")
print(f"{'✅' if has_val_sig else '❌'} IndustryRotationSignal.valuation_signal: {has_val_sig} (V11.5)")

# 验证行业估值权重
iv_weights = sr_engine._industry_val_weights
assert "momentum" in iv_weights, "行业估值权重缺少 momentum"
assert "valuation" in iv_weights, "行业估值权重缺少 valuation"
iv_total = sum(iv_weights.values())
assert abs(iv_total - 1.0) < 0.001, f"行业估值权重之和应为 1.0, 实际 {iv_total}"
print(f"✅ 行业估值双因子权重: momentum={iv_weights['momentum']}, valuation={iv_weights['valuation']}")

# 执行计算
try:
    sr_signal = sr_engine.calculate()
    print(f"\n📊 StyleRotationSignal 计算:")
    print(f"   composite_signal: {sr_signal.composite_signal:.2f}")
    print(f"   composite_direction: {sr_signal.composite_direction}")
    print(f"   data_available: {sr_signal.data_available}")
    print(f"   style_signal: {sr_signal.style_signal:.2f} ({sr_signal.style_direction})")
    print(f"   size_signal: {sr_signal.size_signal:.2f} ({sr_signal.size_direction})")
    print(f"   industry_rotation: {len(sr_signal.industry_rotation)} 行业")
    print(f"   rotation_alerts: {sr_signal.rotation_alerts}")
    
    # 验证行业轮动中是否包含估值因子
    if sr_signal.industry_rotation:
        sample_industry = list(sr_signal.industry_rotation.values())[0]
        print(f"\n   📋 样本行业信号 ({sample_industry.industry}):")
        print(f"      momentum_5d: {sample_industry.momentum_5d:.2f}%")
        print(f"      momentum_20d: {sample_industry.momentum_20d:.2f}%")
        print(f"      pe_ttm: {sample_industry.pe_ttm:.2f} (V11.5)")
        print(f"      pe_percentile: {sample_industry.pe_percentile:.2f} (V11.5)")
        print(f"      valuation_signal: {sample_industry.valuation_signal:.2f} (V11.5)")
        print(f"      signal: {sample_industry.signal:.2f}")
    
    # 验证信号范围
    assert -100 <= sr_signal.composite_signal <= 100
    print(f"\n✅ 信号范围验证通过")
    
except Exception as e:
    print(f"⚠️ StyleRotationEngine 计算失败: {e}")

print("\n🎉 3.3 StyleRotationEngine V11.5 测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3.4 OptionPCREngine — V11 PCR 信号测试
# ═══════════════════════════════════════════════════════════════════════
from subsystems.market_state.core.option_pcr_engine import OptionPCREngine, OptionPCRSignal

print("--- OptionPCREngine V11 测试 ---")

# 初始化 OptionPCREngine
pcr_engine = OptionPCREngine(
    data_service=data_loader,
    config=config,
    cache=cache,
)
print(f"✅ OptionPCREngine 初始化成功")
print(f"   标的: {pcr_engine._underlyings}")
print(f"   权重: {pcr_engine._weights}")
print(f"   阈值: {pcr_engine._thresholds}")

# 验证 PCR 阈值
thresholds = pcr_engine._thresholds
assert thresholds.get("extreme_greed") == 0.5, f"extreme_greed 阈值应为 0.5"
assert thresholds.get("extreme_fear") == 1.5, f"extreme_fear 阈值应为 1.5"
print(f"✅ PCR 阈值: extreme_greed={thresholds['extreme_greed']}, extreme_fear={thresholds['extreme_fear']}")

# 执行计算
try:
    pcr_signal = pcr_engine.calculate()
    print(f"\n📊 OptionPCRSignal 计算:")
    print(f"   composite_signal: {pcr_signal.composite_signal:.2f}")
    print(f"   composite_direction: {pcr_signal.composite_direction}")
    print(f"   data_available: {pcr_signal.data_available}")
    print(f"   volume_pcr: {pcr_signal.volume_pcr:.4f}")
    print(f"   oi_pcr: {pcr_signal.oi_pcr:.4f}")
    print(f"   skew: {pcr_signal.skew:.4f}")
    
    if pcr_signal.data_available:
        # 验证信号范围
        assert -100 <= pcr_signal.composite_signal <= 100
        print(f"✅ 信号范围验证通过")
        
        # PCR 反向指标验证
        print(f"\n   📋 PCR 反向指标逻辑:")
        print(f"      PCR < {thresholds['extreme_greed']} → 极端贪婪 → 看空 (反向)")
        print(f"      PCR > {thresholds['extreme_fear']} → 极端恐惧 → 看多 (反向)")
        print(f"      PCR ∈ [{thresholds['neutral_low']}, {thresholds['neutral_high']}] → 中性")
    else:
        print(f"⚠️ PCR 数据不可用 (期权数据可能未连接)")
    
except Exception as e:
    print(f"⚠️ OptionPCREngine 计算失败: {e}")

# 测试 _pcr_to_signal 方法
try:
    test_pcr_values = [0.3, 0.6, 0.9, 1.0, 1.2, 1.5, 2.0]
    print(f"\n📊 _pcr_to_signal 反向指标映射:")
    for pcr_val in test_pcr_values:
        sig = pcr_engine._pcr_to_signal(pcr_val)
        direction = "看多(反向)" if sig > 20 else ("看空(反向)" if sig < -20 else "中性")
        print(f"   PCR={pcr_val:.1f} → signal={sig:.1f} ({direction})")
    print(f"✅ PCR反向指标映射验证通过")
except Exception as e:
    print(f"⚠️ _pcr_to_signal 测试失败: {e}")

print("\n🎉 3.4 OptionPCREngine 测试完成!")

---
## Module 4: MarketStateEngine V11.5 7-Component Integration Test

### 7分量模型

| # | 分量 | 权重 | 引擎 |
|---|------|------|------|
| 1 | commodity | 0.20 | DerivativesSignalEngine |
| 2 | term_structure | 0.06 | DerivativesSignalEngine |
| 3 | index_basis | 0.16 | DerivativesSignalEngine |
| 4 | **fund_flow** | **0.10** | **FundFlowEngine** |
| 5 | option_pcr | 0.10 | OptionPCREngine |
| 6 | macro_valuation | 0.10 | MacroValuationEngine |
| 7 | style_rotation | 0.28 | StyleRotationEngine |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4.1 MarketStateEngine V11.5 — Bootstrap full system with ServiceContainer
# ═══════════════════════════════════════════════════════════════════════
from base_service.service_container import ServiceContainer
from subsystems.market_state.core.market_state_engine import MarketStateEngine

print("--- MarketStateEngine V11.5 7分量系统集成 ---")

# 创建新的 ServiceContainer 并注册所有服务
integration_container = ServiceContainer()
integration_container.register_singleton("config", config)
integration_container.register_singleton("cache", cache)
integration_container.register_singleton("event_bus", bus)

# 注册数据服务
if data_loader is not None:
    integration_container.register_singleton("data_loader", data_loader)
    print("✅ data_loader 已注册")

if tdx is not None:
    integration_container.register_singleton("tdx_adapter", tdx)
    print("✅ tdx_adapter 已注册")

if db_reader is not None:
    integration_container.register_singleton("db_reader", db_reader)
    print("✅ db_reader 已注册")

if ak_adapter is not None:
    integration_container.register_singleton("ak_adapter", ak_adapter)
    print("✅ ak_adapter 已注册")

# 创建 MarketStateEngine
try:
    mse = MarketStateEngine(integration_container)
    print(f"✅ MarketStateEngine V11.5 实例化成功 (7分量模型)")
except Exception as e:
    mse = None
    print(f"❌ MarketStateEngine 实例化失败: {e}")

# 启动 MarketStateEngine
if mse is not None:
    try:
        mse.start()
        print(f"✅ MarketStateEngine V11.5 启动成功")
        print(f"   is_running: {mse.is_running}")
        
        # 验证所有引擎组件
        print(f"\n📋 引擎组件状态:")
        print(f"   ContractManager: {'✅' if mse.contract_manager else '❌'}")
        print(f"   DerivativesSignalEngine: {'✅' if mse.signal_engine else '❌'}")
        print(f"   FundFlowEngine (V11.5): {'✅' if mse.fund_flow_engine else '❌'}")
        print(f"   OptionPCREngine: {'✅' if mse.option_pcr_engine else '❌'}")
        print(f"   MacroValuationEngine (V11.5): {'✅' if mse.macro_valuation_engine else '❌'}")
        print(f"   StyleRotationEngine (V11.5): {'✅' if mse.style_rotation_engine else '❌'}")
        
    except Exception as e:
        print(f"⚠️ MarketStateEngine 启动失败: {e}")

print("\n🎉 4.1 MarketStateEngine V11.5 Bootstrap 完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4.2 MarketStateEngine V11.5 — Execute 7-component calculation
# ═══════════════════════════════════════════════════════════════════════

if mse is not None and mse.is_running:
    print("--- MarketStateEngine V11.5 7分量计算 ---")
    
    try:
        result = mse.run()
        
        if result is not None:
            print(f"\n📊 V11.5 7分量综合结果:")
            print(f"   composite_signal: {result.composite_signal:.2f}")
            print(f"   composite_direction: {result.composite_direction}")
            
            # ── 各分量信号展示 ──────────────────────────────────
            print(f"\n📋 7分量详细信号:")
            
            # 1-3: DerivativesSignalEngine 分量
            if result.commodity_signals:
                avg_commodity = sum(s.signal for s in result.commodity_signals.values()) / len(result.commodity_signals)
                print(f"   1️⃣  commodity (0.20): avg_signal={avg_commodity:.2f}, 品种数={len(result.commodity_signals)}")
            else:
                print(f"   1️⃣  commodity (0.20): 无数据")
            
            if result.term_structure:
                avg_term = sum(s.signal for s in result.term_structure.values()) / len(result.term_structure)
                print(f"   2️⃣  term_structure (0.06): avg_signal={avg_term:.2f}, 品种数={len(result.term_structure)}")
            else:
                print(f"   2️⃣  term_structure (0.06): 无数据")
            
            if result.index_futures_basis:
                avg_basis = sum(s.signal for s in result.index_futures_basis.values()) / len(result.index_futures_basis)
                print(f"   3️⃣  index_basis (0.16): avg_signal={avg_basis:.2f}, 品种数={len(result.index_futures_basis)}")
            else:
                print(f"   3️⃣  index_basis (0.16): 无数据")
            
            # 4: FundFlowEngine (V11.5 REACTIVATED)
            if result.fund_flow_signal and result.fund_flow_signal.data_available:
                ff = result.fund_flow_signal
                print(f"   4️⃣  fund_flow (0.10): signal={ff.composite_signal:.2f} [{ff.composite_direction}] (V11.5 REACTIVATED)")
            else:
                print(f"   4️⃣  fund_flow (0.10): 无数据或不可用")
            
            # 5: OptionPCREngine
            if result.option_pcr_signal and result.option_pcr_signal.data_available:
                pcr = result.option_pcr_signal
                print(f"   5️⃣  option_pcr (0.10): signal={pcr.composite_signal:.2f} [{pcr.composite_direction}]")
            else:
                print(f"   5️⃣  option_pcr (0.10): 无数据或不可用")
            
            # 6: MacroValuationEngine (V11.5 PE-only)
            if result.macro_valuation_signal and result.macro_valuation_signal.data_available:
                mv = result.macro_valuation_signal
                print(f"   6️⃣  macro_valuation (0.10): signal={mv.composite_signal:.2f} [{mv.composite_direction}] (PE only)")
            else:
                print(f"   6️⃣  macro_valuation (0.10): 无数据或不可用")
            
            # 7: StyleRotationEngine (V11.5 dual-factor)
            if result.style_rotation_signal and result.style_rotation_signal.data_available:
                sr = result.style_rotation_signal
                print(f"   7️⃣  style_rotation (0.28): signal={sr.composite_signal:.2f} [{sr.composite_direction}] (dual-factor)")
            else:
                print(f"   7️⃣  style_rotation (0.28): 无数据或不可用")
            
            # 验证综合信号范围
            assert -100 <= result.composite_signal <= 100, f"综合信号超出范围"
            print(f"\n✅ 综合信号范围验证通过: {result.composite_signal:.2f} ∈ [-100, 100]")
            
            # 验证 fund_flow_signal 存在
            assert result.fund_flow_signal is not None, "fund_flow_signal 不应为 None (V11.5)"
            print(f"✅ fund_flow_signal 存在: {type(result.fund_flow_signal).__name__}")
            
        else:
            print(f"⚠️ MarketStateEngine.run() 返回 None")
            
    except Exception as e:
        print(f"⚠️ MarketStateEngine 计算失败: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ MarketStateEngine 未启动, 跳过7分量计算")

print("\n🎉 4.2 MarketStateEngine 7分量计算测试完成!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4.3 MarketStateEngine V11.5 — Generate enhanced report
# ═══════════════════════════════════════════════════════════════════════

if mse is not None and mse.is_running:
    print("--- MarketStateEngine V11.5 Enhanced Report ---")
    
    try:
        report = mse.generate_report()
        
        if report and not report.get("error"):
            # 验证报告版本
            version = report.get("version", "unknown")
            print(f"✅ 报告版本: {version}")
            
            # 验证报告模型
            summary = report.get("summary", {})
            model = summary.get("model", "unknown")
            composite_sig = summary.get("composite_signal", 0)
            composite_dir = summary.get("composite_direction", "unknown")
            print(f"✅ 报告模型: {model}")
            print(f"   composite_signal: {composite_sig}")
            print(f"   composite_direction: {composite_dir}")
            
            # V11.5: 验证 fund_flow section 存在
            has_fund_flow = "fund_flow" in report and report["fund_flow"] is not None
            print(f"{'✅' if has_fund_flow else '❌'} fund_flow section 存在: {has_fund_flow} (V11.5)")
            
            # 验证其他 V11 新增分量
            has_option_pcr = "option_pcr" in report and report["option_pcr"] is not None
            has_macro_val = "macro_valuation" in report and report["macro_valuation"] is not None
            has_style_rot = "style_rotation" in report and report["style_rotation"] is not None
            print(f"{'✅' if has_option_pcr else '⚠️'} option_pcr section: {has_option_pcr}")
            print(f"{'✅' if has_macro_val else '⚠️'} macro_valuation section: {has_macro_val}")
            print(f"{'✅' if has_style_rot else '⚠️'} style_rotation section: {has_style_rot}")
            
            # V11.5: 验证 fund_flow 数据不包含 akshare fund flow 相关信息
            if has_fund_flow:
                ff_section = report["fund_flow"]
                ff_str = str(ff_section)
                has_akshare_ff = 'ak.stock_individual_fund_flow' in ff_str or 'ak.stock_market_fund_flow' in ff_str
                print(f"{'✅' if not has_akshare_ff else '❌'} fund_flow 不含 akshare fund flow API引用: {not has_akshare_ff}")
            
            # 验证 macro_valuation 为 PE-only
            if has_macro_val:
                mv_section = report["macro_valuation"]
                mv_str = str(mv_section)
                has_pb_in_report = 'pb_percentile' in mv_str or '"pb"' in mv_str
                print(f"{'✅' if not has_pb_in_report else '❌'} macro_valuation 无 PB 字段: {not has_pb_in_report} (V11.5 PE-only)")
            
            # 验证 style_rotation 包含估值因子
            if has_style_rot:
                sr_section = report["style_rotation"]
                sr_str = str(sr_section)
                has_val_factor = 'pe_ttm' in sr_str or 'pe_percentile' in sr_str or 'valuation_signal' in sr_str
                print(f"{'✅' if has_val_factor else '⚠️'} style_rotation 包含估值因子: {has_val_factor} (V11.5 dual-factor)")
            
            # 报告结构概览
            print(f"\n📋 报告顶级键: {list(report.keys())}")
            
        else:
            error = report.get("error", "unknown")
            print(f"⚠️ 报告生成错误: {error}")
            
    except Exception as e:
        print(f"⚠️ 报告生成失败: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ MarketStateEngine 未启动, 跳过报告生成")

print("\n🎉 4.3 MarketStateEngine V11.5 Enhanced Report 测试完成!")

---
## Module 5: V11.5 Summary

### V11 vs V11.5 对比表

| 维度 | V11 | V11.5 |
|------|-----|-------|
| 分量数 | 6 (fund_flow DEPRECATED) | **7 (fund_flow REACTIVATED)** |
| fund_flow 数据源 | akshare fund flow API (DEPRECATED) | **TDX成交量数据** |
| 估值指标 | PE + PB | **PE only (no PB)** |
| 估值数据源 | PostgreSQL only | **online-first (AK → PG fallback)** |
| 行业轮动 | momentum 单因子 | **momentum+valuation dual-factor** |
| PE百分位窗口 | - | **window=2500** |
| 数据库列名 | 中文 | 中文 (日期, 滚动市盈率) |

### V11.5 关键变更清单
1. **FundFlowEngine REACTIVATED**: 从DEPRECATED升级为正式分量, 权重0.10
   - 数据源: 仅使用TDX成交量数据
   - 禁止: ak.stock_individual_fund_flow() / ak.stock_market_fund_flow()
   - 4个子信号: volume_trend(0.30), size_rotation(0.30), leverage_ratio(0.20), momentum(0.20)

2. **MacroValuationEngine PE-only**: 移除PB相关逻辑
   - 移除: pb, pb_percentile, has_pb 字段
   - 保留: pe_ttm, pe_percentile (从历史PE序列计算)
   - online-first: AKAdapter.get_index_pe_csindex() → PostgreSQL fallback

3. **StyleRotationEngine dual-factor**: 行业轮动含PE估值
   - IndustryRotationSignal 新增: pe_ttm, pe_percentile, valuation_signal
   - industry_valuation_weights: momentum=0.60, valuation=0.40
   - 行业PE数据: 在线优先(AK) → PG降级

### V11.5 验证清单
- [x] composite_weights 包含7个键 (含 fund_flow=0.10)
- [x] 权重之和 = 1.00
- [x] fund_flow_weights 包含4个子信号
- [x] industry_valuation_weights 包含 momentum + valuation
- [x] PE percentile 配置: source=pe_ttm, window=2500, method=rank
- [x] DatabaseReader: PE-only (无PB字段)
- [x] MacroValuationSignal: 无PB字段, 含pe_percentile_avg
- [x] IndustryRotationSignal: 含pe_ttm, pe_percentile, valuation_signal
- [x] FundFlowEngine: 不使用akshare fund flow API
- [x] Enhanced Report: 含fund_flow section, version="11.5", model="7-component"